<a href="https://colab.research.google.com/github/wjlee62/Math-381-Wi26/blob/main/Homework_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Homework 8, due Thursday, March 12**

# Problem 1

Consider Problem 3a from the last homework. Let $T$ be the random variable which is the recurrence time of state 50; i.e., the number of steps it takes for the Markov chain starting at state 50 to return to state 50. In the previous homework, you estimated the mean of $T$. Complete the following code to estimate the standard deviation of $T$.

In [27]:
import numpy
import random


# Your code here.
n = 100
num_trials = 10000
samples = []

for _ in range(num_trials):
  X = 50
  t = 0

  t += 1

  r = random.randint(1, n)
  if r <= X:
    X -= 1
  else:
    X += 1

  while X != 50:
    t += 1
    r = random.randint(1, n)
    if r <= X:
      X -= 1
    else:
      X += 1

    samples.append(t)

# Calculates the sample standard deviation of a data set.
answer = numpy.std(samples, ddof=1)
print(answer)

45.14951049629358


Based on your answer, estimate the number of simulation trials you would need to find the mean recurrence time of state 50 within 0.1 precision with at least 95% confidence. What about 0.01 precision?

**Answer and explanation:**
0.1 precision (E = 0.1): N = ((z * sigma)/0.1)^2, where z = 1.96 (95% confidence), sigma = 45.149

N = ((1.96 * 45.149)/0.1)^2 = 783084 simulation trials

0.01 precision (E = 0.01): N = ((z * sigma)/0.01)^2, where z = 1.96 (95% confidence), sigma = 45.149

N = ((1.96 * 45.149)/0.01)^2 = 78308411 simulation trials



# Problem 2

We will consider a generalization of the Markov chain from Problem 1 of the last homework. Suppose we have $n$ balls divided between $k$ containers, where $n$ and $k$ are positive integers.

* Every step, one of the $k$ containers is chosen uniformly at random. Call this container $s$.
* If $s$ does not contain any balls, we do nothing that step.
* If $s$ does contain balls, we choose one of the remaining $k-1$ containers unifomly at random. Call this container $t$.
* We move one ball from container $s$ to container $t$.

We represent a state of this Markov chain by a tuple $(a_1,\dots,a_k)$, where $a_i$ is the number of balls in container $i$.

(a) Suppose $a = (a_1,\dots,a_k)$ and $b = (b_1,\dots,b_k)$ are different states of the Markov chain. What are the possible values for the transition probability from $a$ to $b$?

**Answer and explanation:** The possible values for the transition probability from a to b are 0 or 1/(k(k-1)).

(b) Is the uniform distribution the stationary distribution for this Markov chain?

**Answer and explanation:** Yes, the uniform distribution is the stationary distribution because the transition probabilities between any two connected states are symmetric (Pab = Pba). This symmetry makes the transition matrix doubly stochastic, meaning the total probability flowing "into" any state is equal to the total probability flowing "out" when all states are weighed equally.

(c) Suppose we want to estimate the long-term mean value of $\max(a_1,\dots,a_k)$, the number of balls in the container with the most balls. For example, at state $(1,1,8,4,6)$ this number is 8, and at state $(7,7,2,2,2)$ this number is 7. Complete the simulate() function to do this.

In [16]:
# Changes the current state according to one step of the chain.
# The state is stored as a list of integers.
def step(state):
  k = len(state)
  s = random.randint(0, k-1)
  if state[s] == 0:
    return

  # Rejection sampling to choose t
  t = random.randint(0, k-1)
  while t == s:
    t = random.randint(0, k-1)

  state[s] -= 1
  state[t] += 1
  return


# Simulates the Markov chain.
def simulate(n, k, sample_size, burn):
  # Initial state
  state = [0] * k
  state[0] = n


  # Burn-in stage
  for _ in range(burn):
    step(state)


  # Sampling stage
  samples = []
  for _ in range(sample_size):
    step(state)
    samples.append(max(state))


  # Return the average value of max(state)
  return sum(samples)/len(samples)

First, test your code on some values of n, k **where you know what the correct answer should be**.

In [17]:
answer = simulate(1, 2, 10000, 1000)
print(answer)

1.0


Now test your code for $n = 20$, $k = 5$. It is up to you to decide on an appropriate sample size and burn-in period.

In [18]:
answer = simulate(20, 5, 10000, 5000)
print(answer)

9.652


# Problem 3

We will now convert to chain from Problem 2 to a Metropolis chain with weight function

$$
F(a_1,\dots,a_k) = a_1^2 + a_2^2 + \dots + a_k^2.
$$

Rewrite the step() function below to do this. (We will not be changing the simulate() function.)

In [19]:
def step(state):
  k = len(state)
  s = random.randint(0, k-1)
  if state[s] == 0:
    return

  t = random.randint(0, k-1)
  while t == s:
    t = random.randint(0, k-1)

  F_old = sum(x**2 for x in state)
  F_new = F_old - state[s]**2 - state[t]**2 + (state[s]-1)**2 + (state[t]-1)**2
  alpha = F_new/F_old
  r = random.random()

  if r < alpha:
    state[s] -= 1
    state[t] += 1

  return

Again, test your code on some values of n, k where you know what the correct answer should be.

In [20]:
answer = simulate(2, 2, 100000, 5000)
print(answer)

1.0


Then test your code for $n = 20$, $k = 5$. Your answer should be larger than the corresponding answer in Problem 2.

In [21]:
answer = simulate(20, 5, 100000, 10000)
print(answer)

9.59188
